# Laboratorio 2: Motor de búsqueda semántica

Este notebook desarrolla las secciones 7 a 10 del laboratorio: construcción del corpus, definición de consultas, generación de embeddings y cálculo de similitud coseno.

**Dominio seleccionado:** soporte técnico de una plataforma digital.

## Dependencias

Antes de ejecutar el notebook, el ambiente debe tener instaladas las dependencias indicadas en el laboratorio:

```bash
python -m pip install sentence-transformers numpy scikit-learn
```

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

## Sección 7 — Parte A: Corpus

El corpus contiene 24 oraciones en español relacionadas con problemas, configuraciones y solicitudes de soporte técnico.

In [ ]:
CORPUS = [
    "No puedo iniciar sesión en mi cuenta.",
    "Olvidé mi clave de acceso al sistema.",
    "El usuario desea cambiar su contraseña.",
    "La plataforma muestra un error al iniciar sesión.",
    "La cuenta fue bloqueada por demasiados intentos fallidos.",
    "Necesito recuperar el acceso porque perdí mis credenciales.",
    "El sistema recomienda restablecer la clave de seguridad.",
    "No recibí el correo para recuperar mi contraseña.",
    "Quiero actualizar el correo electrónico asociado a mi perfil.",
    "La aplicación móvil se cierra al abrirla.",
    "La página tarda demasiado tiempo en cargar.",
    "El sistema muestra una pantalla en blanco después de ingresar.",
    "No puedo descargar el archivo desde la plataforma.",
    "El documento cargado supera el tamaño permitido.",
    "El micrófono no funciona durante las videollamadas.",
    "La cámara no es detectada por la aplicación.",
    "No escucho ningún sonido en la reunión virtual.",
    "La conexión se interrumpe constantemente durante la llamada.",
    "Deseo activar las notificaciones en mi teléfono.",
    "Las alertas de la aplicación llegan con retraso.",
    "Necesito cambiar el idioma de la interfaz.",
    "El código de verificación nunca llegó a mi celular.",
    "La plataforma no reconoce mi nombre de usuario.",
    "Quiero eliminar permanentemente mi cuenta.",
]

assert len(CORPUS) >= 20, "El corpus debe contener al menos 20 oraciones."

## Sección 8 — Parte B: Consultas

Las consultas incluyen coincidencias literales, relaciones semánticas sin las mismas palabras, un caso ambiguo y situaciones en las que la búsqueda por palabras clave puede fallar.

In [ ]:
CONSULTAS = [
    "quiero cambiar mi contraseña",       # Coincidencia directa de palabras
    "no logro entrar a mi perfil",        # Relación semántica sin coincidencia directa completa
    "la app desaparece cuando la inicio", # Keyword search puede fallar: la aplicación se cierra
    "tengo problemas con el audio",       # Consulta ambigua: micrófono o sonido
    "nunca me enviaron el número para confirmar mi identidad",  # Código de verificación
    "el sitio funciona muy lento",        # Relación semántica: la página tarda en cargar
]

assert len(CONSULTAS) >= 5, "Se requieren al menos 5 consultas de prueba."

## Sección 9 — Parte C: Generación de embeddings

Se utiliza un modelo multilingüe de `sentence-transformers`. Los vectores se normalizan para que su producto punto equivalga a la similitud coseno.

In [ ]:
MODELO_EMBEDDINGS = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

modelo = SentenceTransformer(MODELO_EMBEDDINGS)

In [ ]:
embeddings_corpus = modelo.encode(
    CORPUS,
    normalize_embeddings=True,
)
embeddings_corpus = np.asarray(embeddings_corpus)

print(f"Corpus: {len(CORPUS)} oraciones")
print(f"Dimensión de embeddings: {embeddings_corpus.shape[1]}")

## Sección 10 — Parte D: Similitud coseno

Como todos los embeddings están normalizados, la similitud coseno se calcula mediante el producto punto entre el vector de la consulta y cada vector del corpus.

In [ ]:
def similitud_coseno(
    embedding_consulta: np.ndarray,
    embeddings_documentos: np.ndarray,
) -> np.ndarray:
    """Calcula la similitud coseno entre una consulta y el corpus."""
    return embeddings_documentos @ embedding_consulta

In [ ]:
# Ejemplo de cálculo para la primera consulta.
embedding_consulta = modelo.encode(
    CONSULTAS[0],
    normalize_embeddings=True,
)
puntajes_similitud = similitud_coseno(
    embedding_consulta,
    embeddings_corpus,
)

print(f"Consulta: {CONSULTAS[0]}")
print(f"Cantidad de puntajes calculados: {len(puntajes_similitud)}")
print(puntajes_similitud)